In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import cv2
import os
from pathlib import Path
from tqdm import tqdm

# 疑似的なRAW画像作成
def simulate_RAW(image, h=4, w=4): # h=1/4(画素内輝度変換 x エッジ抽出)　w=1/4(画素内輝度変換 x ビニング)
    # 色ごとの吸収率
    R_abs = 0.25
    G_abs = 0.35
    B_abs = 0.80

    # FHD相当にリサイズ
    image = image.astype(np.float64)
    resize_img = cv2.resize(image, dsize=None, fx=3, fy=3, interpolation=cv2.INTER_LANCZOS4)
    if resize_img.shape[0] % h:
        resize_img = cv2.copyMakeBorder(resize_img, top=0, bottom=h - (resize_img.shape[0] % h), left=0, right=0, borderType=cv2.BORDER_REPLICATE)
    if resize_img.shape[1] % w:
        resize_img = cv2.copyMakeBorder(resize_img, top=0, bottom=0, left=w - (resize_img.shape[1] % w), right=0, borderType=cv2.BORDER_REPLICATE)
    
    R, G, B = resize_img[:,:,2] * R_abs, resize_img[:,:,1] * G_abs, resize_img[:,:,0] * B_abs 
    return R, G, B

# 画素内輝度変換
def Raw_to_gray(R, G, B): 
    # 10bitの分解能でグレイスケール情報を読みだす
    color_filters = {
    'RGGB' : 2.28 * np.array(R[::2,::2] + G[::2,1::2] + G[1::2,::2] + B[1::2,1::2]) #
    }
    gray_img = color_filters['RGGB']

    return gray_img

# ノイズ付加
def noise(image, scale=2):
    noise = np.random.normal(loc=0, scale=scale, size=image.shape)
    noise_image = image + noise
    return noise_image

# 切り捨てビット
def shift_bit(image, input_bit=8, output_bit=4):
    import sys
    if not 1 <= output_bit <= input_bit:
        print(f'Error: The value **output_bit** is 1~{input_bit} bit of int type')
        sys.exit()    
    br_img = image >> (input_bit - output_bit)
    return br_img

# 強調ビット
def emphasis_bit(image, input_bit=8, output_bit=4):
    import sys
    if not 1 <= output_bit <= input_bit:
        print(f'Error: The value **output_bit** is 1~{input_bit} bit of int type')
        sys.exit()
    r = 2**output_bit - 1
    input_depth = 2**input_bit         
    look_up_table = np.zeros((input_depth,1), dtype=np.uint16)
    for i in range(input_depth):
        if ((input_depth >> 1) - r//2) <= i < ((input_depth >> 1) + r//2):
            look_up_table[i] = (i + 1) - ((input_depth >> 1) - r//2)
        elif i < ((input_depth >> 1) - r//2):
            look_up_table[i] = 0
        else:
            look_up_table[i] = r
    return look_up_table[image]

# 水平方向のエッジ抽出 (縦1次微分フィルタ)
def vdiff_2x1(image, output_bit = 11):
    MAX = 2**output_bit
    delta = MAX // 2
    diff_img = image[0::2, 0::1] - image[1::2, 0::1]
    diff_img = diff_img + delta
    diff_img[diff_img < 0], diff_img[diff_img > MAX-1] = 0, MAX-1
    diff_img = diff_img.astype(np.uint16)
    return diff_img

# 水平方向のビニング
def hadd_1x2(image, input_bit = 8):
    MAX = 2**(input_bit + 1)
    
    hadd_img = image[0::1, 0::2] + image[0::1, 1::2]
    hadd_img[hadd_img < 0], hadd_img[hadd_img > MAX-1] = 0, MAX-1 #0より小さいものは0,MAX-1より大きいときはMAX-1
    hadd_img = hadd_img >> 1

    return hadd_img

def onlyhadd_1x2(image, input_bit = 8):
    MAX = 2**(input_bit + 1)
    
    hadd_img = image[0::1, 0::2] + image[0::1, 1::2]
    hadd_img[hadd_img < 0], hadd_img[hadd_img > MAX-1] = 0, MAX-1 #0より小さいものは0,MAX-1より大きいときはMAX-1

    return hadd_img

# ビット数削減アルゴリズム
def shift_algorithm(image, input_bit=10, output_bit=8):
    shift_img = shift_bit(image, input_bit=input_bit, output_bit=output_bit)
    shift_img = shift_img.astype(np.uint8)
    shift8_img = shift_img << (8-output_bit)
    return shift8_img

def emphasis_algorithm(image, input_bit=10, output_bit=8):
    emphasis_img = emphasis_bit(image, input_bit=input_bit, output_bit=output_bit)[:,:,0]
    emphasis_img = emphasis_img.astype(np.uint8)
    emphasis8_img = emphasis_img << (8-output_bit)
    return emphasis8_img
    
def mix_algorithm(image, input_bit=11, Decimention=2, Clipping=1):
    middle_bit = input_bit - Decimention
    output_bit = output_bit= input_bit - Decimention - Clipping
    mix_img = emphasis_bit(shift_bit(image, input_bit=input_bit, output_bit=middle_bit), input_bit=middle_bit, output_bit= output_bit)[:,:,0]
    return mix_img

def gen_value2img(img1):
    img8 = img1.astype(np.uint8)*255
    return img8

# 勾配あるかないか
def grad_or_nograd(image, input_bit=8, clipping_bit=4):
    nograd = 0
    grad = 1
    import sys
    if not 1 <= clipping_bit <= input_bit:
        print(f'Error: The value **output_bit** is 1~{input_bit} bit of int type')
        sys.exit()
    r = 2**clipping_bit - 1
    input_depth = 2**input_bit    
    look_up_table = np.zeros((input_depth,1), dtype=np.uint16)
    for i in range(input_depth):
        if ((input_depth >> 1) - r//2) <= i < ((input_depth >> 1) + r//2):
            look_up_table[i] = nograd
        elif i < ((input_depth >> 1) - r//2):
            look_up_table[i] = grad
        else:
            look_up_table[i] = grad
    return look_up_table[image]

# 画像を8bit形式に変換
def to_8bitimg(image, input_bit=8):
    img8 = image.astype(np.uint8) << 8 - input_bit
    return img8

# 画像保存簡略化
def save_image_at_dir(dir_path, filename, image, compression=1):
    os.makedirs(dir_path, exist_ok=True)
    cv2.imwrite(f'{dir_path}/{filename}.png', image, params=[int(cv2.IMWRITE_PNG_COMPRESSION), compression]) 

# ヒストグラム生成簡略化
def plot_hist(image, bit_num=10):
    MAX = 2**bit_num
    plt.hist(image.flatten(), MAX, [0, MAX-1])
    plt.xticks(ticks=[0, MAX//4, MAX//2, MAX*3//4, MAX-1])
    plt.show()

# 画像プロット簡略化
def plot(image, bit_num=8, ch=3):
    MAX = 2**bit_num
    if ch==3:
        img_rgb = image[:, :, [2, 1, 0]]
        plt.imshow(img_rgb)
    elif ch==1:
        plt.imshow(image, vmin=0, vmax=MAX-1, cmap='gray')  

In [ ]:
dataset_dir = "/workspace/hdd/original_8bit"
images_names = ["train2017","val2017"]

## 8bit

In [ ]:
save_dir = "/workspace/YOLOv7/coco/images" # 任意の保存先ディレクトリのパスに書き換える

CLLIPING_BIT = 2
DICIMENTION_BIT = 1

for image_name in images_names:
    image_paths = list()
    image_paths.extend(list(Path(f"{dataset_dir}/{image_name}").glob("*.jpg")))
    if len(image_paths) == 0:
        raise print('Not find dataset')
    image_paths.sort()
        
    with tqdm(total=len(image_paths), desc=f"[{image_name}]") as pbar:
        for i, image_path in enumerate(image_paths):
            img = cv2.imread(str(image_path))
            R, G, B = simulate_RAW(img)
            
            gray_img = Raw_to_gray(R, G, B)
            noise_img = noise(gray_img, scale=2)

            diff_img = vdiff_2x1(noise_img, output_bit=11)
            shift_img = mix_algorithm(diff_img, input_bit=11, Clipping=CLLIPING_BIT, Decimention=DICIMENTION_BIT)
                        
            hadd_img = hadd_1x2(shift_img, input_bit=8)
            out_img = to_8bitimg(hadd_img, input_bit=8)

            save_image_at_dir(dir_path=f"{save_dir}/{image_name}", filename=image_path.stem, image=out_img)
            
            pbar.update(1)

## 3bit

In [ ]:
save_dir = "/workspace/YOLOv7/coco/images" # 任意の保存先ディレクトリのパスに書き換える

CLLIPING_BIT = 4
DICIMENTION_BIT = 4

for image_name in images_names:
    image_paths = list()
    image_paths.extend(list(Path(f"{dataset_dir}/{image_name}").glob("*.jpg")))
    if len(image_paths) == 0:
        raise FileNotFoundError('Dataset not found at the specified path.')
    image_paths.sort()
        
    with tqdm(total=len(image_paths), desc=f"[{image_name}]") as pbar:
        for i, image_path in enumerate(image_paths):
            img = cv2.imread(str(image_path))
            R, G, B = simulate_RAW(img)
            
            gray_img = Raw_to_gray(R, G, B)
            noise_img = noise(gray_img, scale=2)

            diff_img = vdiff_2x1(noise_img, output_bit=11)
            shift_img = mix_algorithm(diff_img, input_bit=11, Clipping=CLLIPING_BIT, Decimention=DICIMENTION_BIT)
                        
            hadd_img = hadd_1x2(shift_img, input_bit=3)
            out_img = to_8bitimg(hadd_img, input_bit=3)

            save_image_at_dir(dir_path=f"{save_dir}/{image_name}", filename=image_path.stem, image=out_img)
            
            pbar.update(1)

# 1bit

In [ ]:
save_dir = "/workspace/YOLOv7/coco/images" # 任意の保存先ディレクトリのパスに書き換える

CLLIPING_BIT = 6

for image_name in images_names:
    image_paths = list()
    image_paths.extend(list(Path(f"{dataset_dir}/{image_name}").glob("*.jpg")))
    if len(image_paths) == 0:
        raise print('Not find dataset')
    image_paths.sort()
        
    with tqdm(total=len(image_paths), desc=f"[{image_name}]") as pbar:
        for i, image_path in enumerate(image_paths):
            img = cv2.imread(str(image_path))
            R, G, B = simulate_RAW(img)
            
            gray_img = Raw_to_gray(R, G, B)
            noise_img = noise(gray_img, scale=2)

            diff_img = vdiff_2x1(noise_img, output_bit=11)
            grad_img = grad_or_nograd(diff_img, input_bit=11, clipping_bit=CLLIPING_BIT)
            hadd_img = onlyhadd_1x2(grad_img, input_bit=1)
            hadd_img = hadd_img >> 1
            out_img = gen_value2img(hadd_img)

            save_image_at_dir(dir_path=f"{save_dir}/{image_name}", filename=image_path.stem, image=out_img)
            
            pbar.update(1)